一个典型的多轮对话，在 Qwen2.5 的模板下，会被格式化成类似下面的字符串（其中 `<|im_start|>` 和 `<|im_end|>` 是特殊 token，用于界定角色和内容的边界）：

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
你好，请介绍一下你自己。<|im_end|>
<|im_start|>assistant
你好，我是通义千问...<|im_end|>
```
这可以确保模型能正确理解“谁在说话”、“指令是什么”、“应该从哪里开始回答”。<br>
在投入 GPU 资源开始训练之前，必须基于对模型的理解来制定微调策略。这能帮助我们明确配置、正确处理数据，并最大化微调效果。整个策略的制定，可以围绕以下四个问题展开。

（1）首先需要明确最基础的模型架构。Qwen2.5 是一个标准的 **Decoder-only** 架构，唯一的预训练任务就是 **从左到右预测下一个词 (Causal Language Modeling)**。这一“天性”决定了我们的微调必须遵循它的规则。在配置 `LoraConfig` 时，`task_type` 参数必须设为 `"CAUSAL_LM"`；同时，所有微调数据都必须被构造成 `[输入/提示] -> [期望的输出]` 的因果序列格式，让模型学习在给定“因”之后，生成我们期望的“果”。

（2）要确保我们与模型之间的“沟通语言”是正确无误的。模型通过**分词器（Tokenizer）**和**对话模板（Chat Template）**来理解输入。任何偏差都会导致灾难性的误解。因此，数据预处理的核心，就是将我们的原始数据，如问答对，**严格地转换** 为 Qwen2.5 的官方对话模板格式。这不仅是简单的字符串拼接，而是要使用 `<|im_start|>` 和 `<|im_end|>` 等特殊 Token，精确地构建出模型能够理解的结构化内容。最可靠的方法是调用 `transformers` 库提供的 `apply_chat_template` 函数，它能保证格式的绝对正确。

（3）我们需要决定将 LoRA 适配器“注入”到模型的哪些部位。为了最高效地影响模型的行为，我们应将目标对准计算最密集的“核心组件”，也就是**自注意力**和**前馈网络**模块。这要求我们在 `LoraConfig` 中，将 `target_modules` 参数优先指向这些模块中的线性层，例如注意力模块中的查询、键、值、输出投影层（`q_proj`, `k_proj`, `v_proj`, `o_proj`），以及前馈网络中的门控、上行和下行投影层（`gate_proj`, `up_proj`, `down_proj`）。

（4）微调策略的选择应充分利用模型的“先天优势”。微调并非从零开始，而是对一位博学的“通才”进行专业“深造”。Qwen2.5 在海量数据上进行了预训练，并在**知识、代码和数学**等领域得到了特别强化。这意味着模型已具备了强大的先验知识。如果我们的微调目标恰好与这些领域相关，例如构建一个特定领域的代码助手或知识问答机器人，那么 Qwen2.5 便是一个绝佳的起点。此时，微调过程将事半功倍，因为我们是在模型已有的强大能力之上，引导它去适应我们特定的任务需求。

通过以上四个方面的考量，我们对 Qwen2.5 的微调有了一个初步的规划。

In [1]:
import os
import ssl
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CURL_CA_BUNDLE"] = ""  # 禁用 cert 验证
ssl._create_default_https_context = ssl._create_unverified_context

# 然后正常加载模型
from transformers import AutoModelForCausalLM, BitsAndBytesConfig,AutoTokenizer
import torch
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)
bnb_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
)

def chat(user_message,system_message="你是《黑神话：悟空》领域助手，回答准确、简明。"):
    messages = [
        {"role":"system","content":system_message},
        {"role":"user","content":user_message}  
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generation_ids = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 256
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generation_ids)
    ]
    
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return response

question1 = "我该怎么成为天命人？"
answer1 = chat(question1)
print("Q:", question1)
print("A:", answer1)

question2 = "如何获得并合成出云棍？"
answer2 = chat(question2)
print(f"问题: {question2}\n回答:\n{answer2}")

d:\deeplearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 339/339 [00:29<00:00, 11.54it/s, Materializing param=model.norm.weight]                              


Q: 我该怎么成为天命人？
A: 在《黑神话：悟空》的设定中，成为天命人需要满足特定条件，通常是主角孙悟空的转世或具有特殊血脉。具体步骤包括：

1. 了解游戏背景和世界观。
2. 完成游戏任务，揭示你的身份和背景。
3. 通过特定试炼或挑战证明自己的潜力和意志力。

注意，游戏中的具体流程可能有所不同，请根据游戏实际内容进行操作。
问题: 如何获得并合成出云棍？
回答:
在《黑神话：悟空》中，云棍是通过击败“云”这个BOSS获得的。具体步骤如下：

1. 完成主线任务“桃源”，解锁“云”BOSS。
2. 在特定的地图区域（如天空之城），与“云”进行战斗。
3. 击败“云”后，会掉落云棍。

需要注意的是，《黑神话：悟空》目前处于开发阶段，上述信息基于游戏设定和已公布的内容，实际游戏中的流程可能会有所不同。


In [1]:
import os,torch,json,glob
import ssl
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CURL_CA_BUNDLE"] = ""  # 禁用 cert 验证
ssl._create_default_https_context = ssl._create_unverified_context

max_seq_length = 2048

from transformers import AutoModelForCausalLM, BitsAndBytesConfig,AutoTokenizer
from datasets import load_dataset
model_id = "Qwen/Qwen2.5-7B-Instruct"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

train_files = glob.glob("./wukong_dataset_*.jsonl")
train_set = load_dataset("json", data_files=train_files,split="train")
tokenizer = AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
model = prepare_model_for_kbit_training(base_model)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none", 
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
peft_model = get_peft_model(model, lora_cfg)

def format_sample_for_qwen(record):
    instr = record.get("instruction")
    ans = record.get("output")
    
    # 严格类型检查
    if not isinstance(instr, str) or not isinstance(ans, str):
        return {"input_ids": [], "labels": []}
    
    instr = instr.strip()
    ans = ans.strip()
    if not instr or not ans:
        return {"input_ids": [], "labels": []}

    try:
        # 构造完整对话（含 assistant）
        messages_full = [
            {"role": "system", "content": "你是《黑神话：悟空》领域助手，回答准确、简明。"},
            {"role": "user", "content": instr},
            {"role": "assistant", "content": ans}
        ]
        full_text = tokenizer.apply_chat_template(
            messages_full,
            tokenize=False,  # ← 关键：只返回字符串
            add_generation_prompt=False
        )

        # 构造 prompt（不含 assistant 回答）
        messages_prompt = messages_full[:-1]
        prompt_text = tokenizer.apply_chat_template(
            messages_prompt,
            tokenize=False,
            add_generation_prompt=True  # 添加 assistant 起始标记
        )

        # 手动 tokenize
        full_ids = tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_seq_length
        )["input_ids"]

        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_seq_length
        )["input_ids"]

        # 构造 labels
        cut = min(len(prompt_ids), len(full_ids))
        labels = [-100] * cut + full_ids[cut:]
        labels = labels[:max_seq_length]

        return {
            "input_ids": full_ids[:max_seq_length],
            "labels": labels
        }

    except Exception:
        return {"input_ids": [], "labels": []}
train_set = train_set.map(format_sample_for_qwen, remove_columns=["instruction","output"])
train_set = train_set.filter(
    lambda x: isinstance(x["input_ids"], list) 
              and isinstance(x["labels"], list)
              and len(x["input_ids"]) > 0
)

from typing import Dict, List
class DataCollatorForQwen:
    def __init__(self,pad_id:int,max_length:int=2048, ignore_id:int=-100):
        self.pad_id = pad_id
        self.max_length = max_length
        self.ignore_id = ignore_id
    def __call__(self,features:List[Dict]):
        max_len = max(len(f["input_ids"]) for f in features)
        max_len = min(max_len, self.max_length)
        inputs_ids ,labels = [],[]
        for f in features:
            ids = f["input_ids"][:max_len]
            ibs = f["labels"][:max_len]
            padding_length = max_len - len(ids)
            ids = ids + [self.pad_id] * padding_length
            ibs = ibs + [self.ignore_id] * padding_length
            inputs_ids.append(torch.tensor(ids,dtype=torch.long))
            labels.append(torch.tensor(ibs,dtype=torch.long))
        return {
            "input_ids": torch.stack(inputs_ids),
            "labels": torch.stack(labels),
        }
collator = DataCollatorForQwen(
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    max_length = max_seq_length,
    ignore_id = -100,
)

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
args = TrainingArguments(
    output_dir = "./qwen2.5-wukong-qlora",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 1e-3,
    num_train_epochs = 3,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,
    logging_steps = 1,
    optim = "adamw_torch",
    bf16=True,
)

trainer = Trainer(
    model = peft_model,
    train_dataset = train_set,
    args = args,
    data_collator = collator,
)
train_output = trainer.train()
peft_model.save_pretrained("./qwen2.5-wukong-qlora")

peft_model.eval()
TEST_QUERIES = [
    "我该怎么成为天命人？",
    "如何获得并合成出云棍？",
]
@torch.no_grad()
def chat(user_message,system_message="你是《黑神话：悟空》领域助手，回答准确、简明。"):
    message = [
        {"role":"system","content":system_message},
        {"role":"user","content":user_message}
    ]
    text = tokenizer.apply_chat_template(message,tokenize=True,add_generation_prompt=True,return_tensors="pt")
    input_ids = text["input_ids"].to(peft_model.device)

    generation_ids = peft_model.generate(
        input_ids = input_ids,
        max_new_tokens = 256,
        do_sample = True,
        top_p = 0.9,
        temperature = 0.7,
        repetition_penalty = 1.2,
        eos_token_id = tokenizer.eos_token_id,
        pad_token_id = tokenizer.pad_token_id,
    )
    output_ids = generation_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(output_ids,skip_special_tokens=True)
    return response

for query in TEST_QUERIES:
    answer = chat(query)
    print(f"问题: {query}\n回答:\n{answer}\n{'-'*40}\n")

c:\Users\zhuxu\anaconda3\envs\bipo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bin c:\Users\zhuxu\anaconda3\envs\bipo\lib\site-packages\bitsandbytes\libbitsandbytes_cuda118.dll


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 